# Robotics Homework Notebook

This notebook is a simple homework template for a robotics class.

It has three sections:
1. `DH`: fill in the robot DH/MDH table
2. `FK`: implement the forward kinematics function
3. `IK`: implement the inverse kinematics function

The helper-function cell near the top is provided by the instructor. Run it once and keep it folded. You only need to work in the cells marked with `# TODO`.


In [ ]:
import math
import numpy as np

# Standard MDH parameters used for grading
STANDARD_DH_MATRIX = np.array([
    [0.0,   0.0,   96.2,   0.0],
    [90.0,  96.5,  0.0,   90.0],
    [0.0,   346.5, 0.0,    0.0],
    [0.0,   150.0, 0.0,    0.0],
    [90.0,  0.0,   96.5,  90.0],
    [90.0,  0.0,   185.0, 90.0],
], dtype=float)

POSITION_THRESHOLD_MM = 2.0
ORIENTATION_THRESHOLD_DEG = 1.0

DH_TEST_CASES = {
    "zero_pose": np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0]),
    "typical_pose": np.array([25.0, -35.0, 55.0, 15.0, 35.0, -20.0]),
    "check_pose": np.array([-40.0, 30.0, -60.0, 20.0, -30.0, 45.0]),
}

IK_REFERENCE_JOINT_DEG = np.array([25.0, -35.0, 55.0, 15.0, 35.0, -20.0])


def deg_to_rad(value):
    return math.radians(float(value))


def rad_to_deg(value):
    return math.degrees(float(value))


def wrap_angle_deg(value):
    normalized = (float(value) + 180.0) % 360.0
    return normalized - 180.0


def rot_x(rad):
    c, s = math.cos(rad), math.sin(rad)
    return np.array([[1.0, 0.0, 0.0], [0.0, c, -s], [0.0, s, c]], dtype=float)


def rot_y(rad):
    c, s = math.cos(rad), math.sin(rad)
    return np.array([[c, 0.0, s], [0.0, 1.0, 0.0], [-s, 0.0, c]], dtype=float)


def rot_z(rad):
    c, s = math.cos(rad), math.sin(rad)
    return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]], dtype=float)


def matrix_to_rpy_deg(matrix3):
    r20 = float(matrix3[2, 0])
    if abs(r20) < 0.999999:
        pitch = math.asin(-r20)
        roll = math.atan2(matrix3[2, 1], matrix3[2, 2])
        yaw = math.atan2(matrix3[1, 0], matrix3[0, 0])
    else:
        pitch = math.pi / 2 if r20 <= -0.999999 else -math.pi / 2
        roll = math.atan2(-matrix3[0, 1], matrix3[1, 1])
        yaw = 0.0
    return np.array([rad_to_deg(roll), rad_to_deg(pitch), rad_to_deg(yaw)])


def matrix_to_pose(matrix4):
    rpy = matrix_to_rpy_deg(matrix4[:3, :3])
    return {
        "x": float(matrix4[0, 3]),
        "y": float(matrix4[1, 3]),
        "z": float(matrix4[2, 3]),
        "roll": float(rpy[0]),
        "pitch": float(rpy[1]),
        "yaw": float(rpy[2]),
    }


def mdh_single_transform(theta_rad, alpha_rad, a_mm, d_mm):
    c_theta, s_theta = math.cos(theta_rad), math.sin(theta_rad)
    c_alpha, s_alpha = math.cos(alpha_rad), math.sin(alpha_rad)
    return np.array([
        [c_theta, -s_theta, 0.0, a_mm],
        [s_theta * c_alpha, c_theta * c_alpha, -s_alpha, -s_alpha * d_mm],
        [s_theta * s_alpha, c_theta * s_alpha, c_alpha, c_alpha * d_mm],
        [0.0, 0.0, 0.0, 1.0],
    ], dtype=float)


def forward_kinematics_from_dh_matrix(joint_deg, dh_matrix):
    joint_deg = np.asarray(joint_deg, dtype=float)
    dh_matrix = np.asarray(dh_matrix, dtype=float)
    transform = np.eye(4, dtype=float)
    for i in range(6):
        alpha_deg, a_mm, d_mm, offset_deg = dh_matrix[i]
        theta_rad = deg_to_rad(joint_deg[i] + offset_deg)
        alpha_rad = deg_to_rad(alpha_deg)
        transform = transform @ mdh_single_transform(theta_rad, alpha_rad, a_mm, d_mm)
    return transform


def compute_pose_error(student_matrix, reference_matrix):
    student_pose = matrix_to_pose(student_matrix)
    reference_pose = matrix_to_pose(reference_matrix)
    dx = student_pose["x"] - reference_pose["x"]
    dy = student_pose["y"] - reference_pose["y"]
    dz = student_pose["z"] - reference_pose["z"]
    delta_rotation = student_matrix[:3, :3] @ reference_matrix[:3, :3].T
    trace_value = max(-1.0, min(1.0, (np.trace(delta_rotation) - 1.0) / 2.0))
    orientation_norm_deg = rad_to_deg(math.acos(trace_value))
    return {
        "dx": dx,
        "dy": dy,
        "dz": dz,
        "droll": wrap_angle_deg(student_pose["roll"] - reference_pose["roll"]),
        "dpitch": wrap_angle_deg(student_pose["pitch"] - reference_pose["pitch"]),
        "dyaw": wrap_angle_deg(student_pose["yaw"] - reference_pose["yaw"]),
        "position_norm_mm": math.sqrt(dx * dx + dy * dy + dz * dz),
        "orientation_norm_deg": orientation_norm_deg,
    }


def is_pass(error):
    return (
        error["position_norm_mm"] < POSITION_THRESHOLD_MM
        and error["orientation_norm_deg"] < ORIENTATION_THRESHOLD_DEG
    )


IK_TARGET_MATRIX = forward_kinematics_from_dh_matrix(IK_REFERENCE_JOINT_DEG, STANDARD_DH_MATRIX)
IK_TARGET_POSE = matrix_to_pose(IK_TARGET_MATRIX)

print("Reference parameters and helper functions are ready.")

## DH Section

### Problem Statement
Fill in the `DH / MDH` table for this 6-axis robot.

Use a `6 x 4` matrix. Each row should follow:

`[alpha_deg, a_mm, d_mm, offset_deg]`

Requirements:
- use `deg` for angles
- use `mm` for lengths
- complete the next submission cell
- then run the DH grading cell

### Submission Area
Write your answer in the next code cell under `dh_matrix  # TODO`.


In [ ]:
# TODO: Write your DH / MDH table here
# Each row should be: [alpha_deg, a_mm, d_mm, offset_deg]

dh_matrix = np.array([
    [0.0,   0.0,   96.2,   0.0],
    [90.0,  96.5,  0.0,   90.0],
    [0.0,   346.5, 0.0,    0.0],
    [0.0,   150.0, 0.0,    0.0],
    [90.0,  0.0,   96.5,  90.0],
    [90.0,  0.0,   185.0, 90.0],
], dtype=float)

print("dh_matrix is set. Run the next cell to grade your answer.")


In [ ]:
# DH grading cell

def evaluate_dh(student_dh_matrix):
    print("DH grading result")
    print("-" * 60)
    all_pass = True
    for case_name, joint_deg in DH_TEST_CASES.items():
        reference_matrix = forward_kinematics_from_dh_matrix(joint_deg, STANDARD_DH_MATRIX)
        student_matrix = forward_kinematics_from_dh_matrix(joint_deg, student_dh_matrix)
        error = compute_pose_error(student_matrix, reference_matrix)
        passed = is_pass(error)
        all_pass = all_pass and passed
        print(f"{case_name}: {'Pass' if passed else 'Fail'}")
        print(f"  position error = {error['position_norm_mm']:.3f} mm")
        print(f"  orientation error = {error['orientation_norm_deg']:.3f} deg")
    print("-" * 60)
    print("DH overall result:", "Pass" if all_pass else "Fail")

evaluate_dh(dh_matrix)


## FK Section

### Problem Statement
Implement the forward kinematics function `forward_kinematics` for this robot.

Inputs:
- `joint_deg`: a 6-element joint angle vector in `deg`
- `dh_matrix`: a `6 x 4` table with rows in the form `[alpha_deg, a_mm, d_mm, offset_deg]`

Output:
- a `4 x 4` homogeneous transform matrix

Requirements:
- complete the function in the next submission cell
- you may use the helper functions above
- then run the FK grading cell

### Submission Area
Complete `forward_kinematics` in the next code cell.


In [ ]:
# TODO: Implement your FK function here

def forward_kinematics(joint_deg, dh_matrix):
    # Replace the line below with your own solution.
    return 

print("forward_kinematics is defined. Run the next cell to grade your answer.")


In [ ]:
# FK grading cell

def evaluate_fk(fk_function, use_dh_matrix):
    print("FK grading result")
    print("-" * 60)
    all_pass = True
    for case_name, joint_deg in DH_TEST_CASES.items():
        student_matrix = fk_function(joint_deg, use_dh_matrix)
        reference_matrix = forward_kinematics_from_dh_matrix(joint_deg, STANDARD_DH_MATRIX)
        error = compute_pose_error(student_matrix, reference_matrix)
        passed = is_pass(error)
        all_pass = all_pass and passed
        print(f"{case_name}: {'Pass' if passed else 'Fail'}")
        print(f"  position error = {error['position_norm_mm']:.3f} mm")
        print(f"  orientation error = {error['orientation_norm_deg']:.3f} deg")
    print("-" * 60)
    print("FK overall result:", "Pass" if all_pass else "Fail")

evaluate_fk(forward_kinematics, dh_matrix)


## IK Section

### Problem Statement
Implement the inverse kinematics function `Inverse_kinematics` for this robot.

The target pose is given as:
- `IK_TARGET_POSE`

Inputs:
- `target_pose`: a dictionary with `x, y, z, roll, pitch, yaw`
- `dh_matrix`: a `6 x 4` table with rows in the form `[alpha_deg, a_mm, d_mm, offset_deg]`

Output:
- a list of candidate joint solutions
- each solution should be a 6-element vector in `deg`
- for example: `[solution1, solution2, ...]`

Notes:
- your solution does not need to match the reference joint angles exactly
- each returned solution will be checked by FK against the target pose
- if the pose error is small enough, the solution will pass

### Submission Area
Complete `Inverse_kinematics` in the next code cell.


In [ ]:
# TODO: Implement your IK function here

def Inverse_kinematics(target_pose, dh_matrix):
    # Replace the line below with your own solution.
    return 

print("Target pose IK_TARGET_POSE =")
print(IK_TARGET_POSE)
print("Inverse_kinematics is defined. Run the next cell to grade your answer.")


In [ ]:
# IK grading cell

def evaluate_ik(ik_function, use_dh_matrix):
    print("IK grading result")
    print("-" * 60)
    solutions = ik_function(IK_TARGET_POSE, use_dh_matrix)
    if solutions is None or len(solutions) == 0:
        print("No solution was returned.")
        print("IK overall result: Fail")
        return

    any_pass = False
    for index, joint_deg in enumerate(solutions, start=1):
        joint_deg = np.asarray(joint_deg, dtype=float)
        student_matrix = forward_kinematics_from_dh_matrix(joint_deg, use_dh_matrix)
        error = compute_pose_error(student_matrix, IK_TARGET_MATRIX)
        passed = is_pass(error)
        any_pass = any_pass or passed
        print(f"solution_{index}: {'Pass' if passed else 'Fail'}")
        print(f"  joint_deg = {joint_deg}")
        print(f"  position error = {error['position_norm_mm']:.3f} mm")
        print(f"  orientation error = {error['orientation_norm_deg']:.3f} deg")
    print("-" * 60)
    print("IK overall result:", "Pass" if any_pass else "Fail")

evaluate_ik(Inverse_kinematics, dh_matrix)
